# 02 — H&M retrieval

Notebook này dùng lịch sử mua hàng, embedding text và ảnh để gợi ý sản phẩm tiếp theo.

## 1. Chuẩn bị môi trường

Đọc cấu hình đã tạo ở bước trước và cố định seed để kết quả dễ lặp lại.

In [ ]:
from pathlib import Path
import json
import os
import random

import numpy as np
import polars as pl
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [ ]:
PROFILE = os.getenv("HM_PROFILE", "quick").lower()
assert PROFILE in {"quick", "full"}
MODALITY = os.getenv("HM_MODALITY", "multimodal").lower()
assert MODALITY in {"id", "text", "image", "multimodal"}

# Kaggle Input chỉ có quyền đọc; chỉ lưu kết quả vào /kaggle/working.
DATA_DIR = Path(os.getenv("HM_DATA_DIR", os.getenv("HM_WORK_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled")))
OUTPUT_DIR = Path(os.getenv("HM_OUTPUT_DIR", "/kaggle/working/hm_retrieval"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = json.loads((DATA_DIR / "run_config.json").read_text())
SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Profile: {PROFILE} | Modality: {MODALITY} | Device: {device}")
print(f"Read from: {DATA_DIR}\nSave to: {OUTPUT_DIR}")

## 2. Đọc dữ liệu và embedding

Thêm một vector toàn 0 ở đầu để dùng làm padding.

In [ ]:
items = pl.read_parquet(DATA_DIR / "items.parquet")
item_ids = items["item_id"].to_list()

text_raw = np.load(DATA_DIR / "embedding/hm/text_embeddings.npy")
image_raw = np.load(DATA_DIR / "embedding/hm/image_embeddings.npy")

text = np.vstack([np.zeros((1, text_raw.shape[1]), dtype="float32"), text_raw])
image = np.vstack([np.zeros((1, image_raw.shape[1]), dtype="float32"), image_raw])

print(f"Số sản phẩm: {len(item_ids):,}")
print(f"Text shape: {text.shape} | Image shape: {image.shape}")

In [ ]:
vocab = {item_id: idx + 1 for idx, item_id in enumerate(item_ids)}

def load_daily_baskets(name):
    data = pl.read_parquet(DATA_DIR / f"{name}.parquet")
    data = data.with_columns(pl.col("article_id").replace_strict(vocab, default=0).alias("idx"))
    data = data.filter(pl.col("idx") > 0)
    daily = data.group_by(["customer_id", "t_dat"]).agg(pl.col("idx").unique().alias("basket"))
    daily = daily.sort(["customer_id", "t_dat"])

    baskets = {}
    for row in daily.to_dicts():
        baskets.setdefault(row["customer_id"], []).append(row["basket"])
    return baskets

def flatten_baskets(baskets):
    return [item for basket in baskets for item in basket]

train_baskets = load_daily_baskets("train")
# Tên file cũ là rerank_train, nhưng ở notebook này nó chỉ là cửa sổ chọn checkpoint.
selection_baskets = load_daily_baskets("rerank_train")
valid_baskets = load_daily_baskets("valid")
test_baskets = load_daily_baskets("test")

train = {user: flatten_baskets(baskets) for user, baskets in train_baskets.items()}
selection_window = {user: flatten_baskets(baskets) for user, baskets in selection_baskets.items()}
valid = {user: flatten_baskets(baskets) for user, baskets in valid_baskets.items()}
test = {user: flatten_baskets(baskets) for user, baskets in test_baskets.items()}

print(f"Train users: {len(train):,} | Valid users: {len(valid):,}")
print(f"Train days: {sum(map(len, train_baskets.values())):,} | Train events: {sum(map(len, train.values())):,}")

## 3. Tạo dữ liệu train và mô hình

Mỗi lịch sử mua hàng tạo ra các cặp: các món trước đó → món mua tiếp theo.

In [ ]:
MAXLEN = 30
D_MODEL = 128
N_NEG = int(os.getenv("HM_NUM_NEGATIVES", 128))

class PairData(Dataset):
    def __init__(self, daily_histories):
        self.rows = []
        for days in daily_histories.values():
            # Người chỉ có một ngày mua vẫn đóng góp tín hiệu co-purchase.
            if len(days) == 1 and len(days[0]) > 1:
                basket = days[0]
                self.rows.extend(([item for item in basket if item != target], target) for target in basket)
                continue
            history = []
            for basket in days:
                # Không học thứ tự giả giữa các món được mua cùng một ngày.
                if history:
                    self.rows.extend((history.copy(), target) for target in basket)
                history.extend(basket)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        history, target = self.rows[index]
        history = ([0] * MAXLEN + history[-MAXLEN:])[-MAXLEN:]
        return torch.tensor(history), torch.tensor(target)

In [ ]:
class Tower(nn.Module):
    def __init__(self):
        super().__init__()
        self.id = nn.Embedding(len(item_ids) + 1, D_MODEL, padding_idx=0)
        self.text_proj = nn.Linear(text.shape[1], D_MODEL, bias=False)
        self.image_proj = nn.Linear(image.shape[1], D_MODEL, bias=False)
        self.image_gate = nn.Parameter(torch.tensor(0.0))
        # ID là residual nhỏ lúc đầu, để không lấn át embedding Jina đã pretrained.
        self.id_gate = nn.Parameter(torch.tensor(-2.0))
        self.register_buffer("text", torch.tensor(text))
        self.register_buffer("image", torch.tensor(image))

    def item_table(self):
        id_vectors = torch.sigmoid(self.id_gate) * self.id.weight
        text_vectors = self.text_proj(self.text)
        image_vectors = self.image_proj(self.image)
        if MODALITY == "id":
            vectors = id_vectors
        elif MODALITY == "text":
            vectors = text_vectors
        elif MODALITY == "image":
            vectors = image_vectors
        else:
            vectors = id_vectors + text_vectors + torch.sigmoid(self.image_gate) * image_vectors
        return nn.functional.normalize(vectors, dim=1)

    def query(self, history, table=None):
        table = self.item_table() if table is None else table
        mask = history.ne(0)
        weights = torch.arange(1, history.shape[1] + 1, device=history.device)[None] * mask
        vectors = table[history]
        query = (vectors * weights[:, :, None]).sum(1) / weights.sum(1, keepdim=True).clamp_min(1)
        return nn.functional.normalize(query, dim=1)


In [ ]:
freq = np.ones(len(item_ids) + 1, dtype="float64")
freq[0] = 0  # PAD không được lấy làm negative.
for sequence in train.values():
    freq[np.asarray(sequence)] += 1
popular_prob = freq ** 0.75
popular_prob /= popular_prob.sum()
uniform_prob = np.zeros_like(popular_prob)
uniform_prob[1:] = 1 / len(item_ids)
uniform_ratio = float(os.getenv("HM_UNIFORM_NEG_RATIO", "0.0"))
assert 0 <= uniform_ratio <= 1
negative_prob = uniform_ratio * uniform_prob + (1 - uniform_ratio) * popular_prob
sampling_prob = torch.tensor(negative_prob, dtype=torch.float32, device=device)
popular_ranked = [int(idx) for idx in np.argsort(-freq) if idx != 0]
# Rejection sampling theo history làm q thực tế có điều kiện; tắt Log-Q mặc định và chạy ablation riêng nếu cần.
USE_LOGQ = os.getenv("HM_LOGQ_CORRECTION", "0") == "1"

dataset = PairData(train_baskets)
loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=2, pin_memory=device == "cuda")
model = Tower().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"Train pairs: {len(dataset):,} | Catalog: {len(item_ids):,} | Log-Q: {USE_LOGQ} | Uniform negatives: {uniform_ratio:.0%}")

## 4. Huấn luyện và chọn checkpoint

Dùng cửa sổ chọn checkpoint (đọc từ file `rerank_train.parquet`) để chọn model tốt nhất. Validation và test chưa được dùng ở bước này.

In [ ]:
@torch.no_grad()
def score_metrics(context, targets, ks=(12, 50, 100)):
    model.eval()
    ks = tuple(sorted(set(ks)))
    aps = {k: [] for k in ks}
    recalls = {k: [] for k in ks}
    hit_rates = {k: [] for k in ks}
    table = model.item_table()

    for user, truth in targets.items():
        if not truth or user not in context:
            continue
        history = context[user][-MAXLEN:]
        batch = torch.tensor([[0] * (MAXLEN - len(history)) + history], device=device)
        scores = (model.query(batch, table) @ table.T).squeeze()
        scores[0] = -torch.inf
        ranked = torch.topk(scores, max(ks)).indices.cpu().tolist()

        relevant = set(truth)
        hits = [int(item in relevant) for item in ranked]
        for k in ks:
            hits_k = hits[:k]
            ap = sum(sum(hits_k[:pos + 1]) / (pos + 1) * hit for pos, hit in enumerate(hits_k))
            aps[k].append(ap / min(k, len(relevant)))
            recalls[k].append(sum(hits_k) / len(relevant))
            hit_rates[k].append(float(any(hits_k)))

    result = {"users": len(next(iter(aps.values()))) if aps else 0}
    for k in ks:
        result[f"MAP@{k}"] = float(np.mean(aps[k]) if aps[k] else 0)
        result[f"Recall@{k}"] = float(np.mean(recalls[k]) if recalls[k] else 0)
        result[f"HitRate@{k}"] = float(np.mean(hit_rates[k]) if hit_rates[k] else 0)
    return result

@torch.no_grad()
def fair_score_metrics(context, targets, ks=(12, 50, 100, 500, 1000), target_filter=None):
    """Đánh giá next-basket đầy đủ, kể cả user không có history (fallback popularity)."""
    model.eval()
    ks = tuple(sorted(set(ks)))
    values = {name: {k: [] for k in ks} for name in ("map", "recall", "hit", "ndcg")}
    table = model.item_table()
    recommended, fallback_users = set(), 0

    for user, truth in targets.items():
        relevant = set(truth)
        if target_filter is not None:
            relevant = {item for item in relevant if target_filter(user, item)}
        if not relevant:
            continue

        history = context.get(user, [])[-MAXLEN:]
        if history:
            batch = torch.tensor([[0] * (MAXLEN - len(history)) + history], device=device)
            scores = (model.query(batch, table) @ table.T).squeeze()
            scores[0] = -torch.inf
            ranked = torch.topk(scores, max(ks)).indices.cpu().tolist()
        else:
            ranked = popular_ranked[:max(ks)]
            fallback_users += 1

        recommended.update(ranked)
        hits = [int(item in relevant) for item in ranked]
        for k in ks:
            hits_k = hits[:k]
            ap = sum(sum(hits_k[:pos + 1]) / (pos + 1) * hit for pos, hit in enumerate(hits_k))
            dcg = sum(hit / np.log2(pos + 2) for pos, hit in enumerate(hits_k))
            idcg = sum(1 / np.log2(pos + 2) for pos in range(min(k, len(relevant))))
            values["map"][k].append(ap / min(k, len(relevant)))
            values["recall"][k].append(sum(hits_k) / len(relevant))
            values["hit"][k].append(float(any(hits_k)))
            values["ndcg"][k].append(dcg / idcg)

    n_users = len(next(iter(values["map"].values())))
    result = {"evaluated_users": n_users, "fallback_popularity_users": fallback_users,
              f"CatalogCoverage@{max(ks)}": len(recommended) / len(item_ids)}
    for k in ks:
        result[f"MAP@{k}"] = float(np.mean(values["map"][k]) if values["map"][k] else 0)
        result[f"Recall@{k}"] = float(np.mean(values["recall"][k]) if values["recall"][k] else 0)
        result[f"HitRate@{k}"] = float(np.mean(values["hit"][k]) if values["hit"][k] else 0)
        result[f"NDCG@{k}"] = float(np.mean(values["ndcg"][k]) if values["ndcg"][k] else 0)
        result[f"RecallLiftVsRandom@{k}"] = result[f"Recall@{k}"] / (k / len(item_ids))
    return result

In [ ]:
def train_one_epoch():
    model.train()
    losses = []

    for history, target in loader:
        history, target = history.to(device), target.to(device)
        negative = np.random.choice(len(negative_prob), size=(len(target), N_NEG), p=negative_prob)
        # Không dùng positive hiện tại hoặc lịch sử người dùng làm negative.
        target_np = target.cpu().numpy()[:, None]
        history_np = history.cpu().numpy()
        collision = (negative == target_np) | (negative[:, :, None] == history_np[:, None, :]).any(axis=2)
        while collision.any():
            negative[collision] = np.random.choice(len(negative_prob), size=collision.sum(), p=negative_prob)
            collision = (negative == target_np) | (negative[:, :, None] == history_np[:, None, :]).any(axis=2)
        candidates = torch.cat([target[:, None], torch.tensor(negative, device=device)], dim=1)

        table = model.item_table()
        logits = (model.query(history, table)[:, None, :] * table[candidates]).sum(-1) / 0.07
        if USE_LOGQ:
            logits = logits - (N_NEG * sampling_prob[candidates]).clamp_min(1e-12).log()
        labels = torch.zeros(len(target), dtype=torch.long, device=device)
        loss = nn.functional.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses))

In [ ]:
epochs = int(os.getenv("HM_EPOCHS", 8 if PROFILE == "quick" else 20))
best_recall = -1
history = []

# Thêm cấu hình Early Stopping
patience = 3       # Số epoch cho phép metric không cải thiện
patience_counter = 0

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch()
    metrics = score_metrics(train, selection_window)
    row = {"epoch": epoch, "loss": train_loss, **metrics}
    history.append(row)
    print(row)

    if metrics["Recall@100"] > best_recall:
        best_recall = metrics["Recall@100"]
        torch.save({"state": model.state_dict(), "item_ids": item_ids, "d_model": D_MODEL, "valid": metrics}, OUTPUT_DIR / "best_retrieval.pt")
        patience_counter = 0  # Reset lại bộ đếm khi có cải thiện
    else:
        patience_counter += 1
        print(f"Patience counter: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}! Dừng huấn luyện.")
            break  # <-- Dừng vòng lặp sớm ở đây

(OUTPUT_DIR / "retrieval_history.json").write_text(json.dumps(history, indent=2))

## 5. Đánh giá trên validation và test

Nạp lại checkpoint tốt nhất rồi mới đánh giá trên hai tập được giữ riêng.

In [ ]:
state = torch.load(OUTPUT_DIR / "best_retrieval.pt", map_location=device)
model.load_state_dict(state["state"])

validation_context = {user: train.get(user, []) + selection_window.get(user, []) for user in valid}
test_context = {user: train.get(user, []) + selection_window.get(user, []) + valid.get(user, []) for user in test}

train_item_set = {item for history in train.values() for item in history}

result = {
    "modality": MODALITY,
    "selection_split": "retrieval_selection_window",
    "selection_source_file": "rerank_train.parquet",
    "best_selection_window": state["valid"],
    "valid_overall": fair_score_metrics(validation_context, valid),
    "test_overall": fair_score_metrics(test_context, test),
    "test_warm_items": fair_score_metrics(test_context, test, target_filter=lambda _, item: item in train_item_set),
    "test_strict_cold_items": fair_score_metrics(test_context, test, target_filter=lambda _, item: item not in train_item_set),
    "test_repeat_items": fair_score_metrics(test_context, test, target_filter=lambda user, item: item in test_context.get(user, [])),
    "test_explore_items": fair_score_metrics(test_context, test, target_filter=lambda user, item: item not in test_context.get(user, [])),
}

(OUTPUT_DIR / "retrieval_metrics.json").write_text(json.dumps(result, indent=2))
display(result)

**Ghi chú:** checkpoint được chọn theo Recall@100 trên cửa sổ chọn checkpoint (file `rerank_train.parquet`). Validation và test chỉ dùng ở bước đánh giá cuối cùng.